# 15buildstory: the Stage 5 story page (Stage 5, part 2 of 2)

Builds `docs/story.html`: a guided story in eight steps. Each step moves the camera and changes the map,
with a short text in the panel. **Next** and **Back** (or the arrow keys) move between steps; the last step
is a free map to explore.

Like the Stage 3 page, it is written directly with deck.gl, because pydeck cannot move the camera from step to step.
Unlike Stage 3, all data is inside the page, so it opens by double-clicking (no web server needed).

- Input: `hex_story.parquet`, `story_summary.json` (14combine), `districts.geojson`
- Output: `docs/story.html`; links to this page are added to `docs/index.html`, `docs/population.html`,
  `docs/floods.html` and `docs/buses.html`

All visual settings and the story texts are in Section 0, marked `# <-`. Numbers in the texts are written as
`{name}` and filled in from `story_summary.json` (Step 1 prints every available name).

## Section 0: settings

In [113]:
from pathlib import Path

BASE = Path.cwd()
PROCESSED = BASE / "data" / "processed"
DOCS = BASE / "docs"
OUT_HTML = DOCS / "story.html"

# ---- Text ----
PAGE_TITLE = "Klang Valley Reach and Risk: flooded in 2021, served today?"   # <- browser tab
TITLE = "Flooded in 2021, served today?"                                    # <- panel heading
POOR_NOTE = ("Poorly served: a walking score of 2 or less of 5 (Stage 2) and a gap in the Rapid KL network: "
             "fewer than 4 Rapid KL or MRT feeder buses an hour within 400 m in the weekday morning peak, "
             "and no rail station within a 15-minute walk (Stage 3).")     # <- shown under the legend
CAVEATS = [                                                                  # <- always visible (short)
    "The floods are from December 2021; the services and bus timetables are from 2026.",
    "Only Rapid KL and MRT feeder buses have open timetables. Smart Selangor and other buses are missing, "
    "so Klang and Sepang have more buses than shown.",
]
MORE_NOTES_TITLE = "More about the data"                                     # <- click to open
MORE_NOTES = [                                                               # <- inside "More about the data"
    "Flood signal: Sentinel-1 radar images of 20 and 22 December 2021, 2 to 4 days after the peak. Strong: at least 5% "
    "of the hexagon flagged on both days. Possible: on one day; 178 of the 195 possible hexagons come from the "
    "22 December image alone. Radar misses water between tall buildings, so this is a minimum.",
    "Walking score: how many of five everyday services (clinic, school, park, grocery, rail station) are within "
    "15 minutes' walk on the OpenStreetMap footpath network. Clinics outside KL may be missing in OpenStreetMap.",
    "Circles on the map mark the story places; their numbers count the flagged hexagons whose centre lies inside the circle.",
    "Kontur underestimates Putrajaya's population (about 8,600 vs about 110,000 in the 2020 census).",
]
CREDIT = ("Radar: Copernicus Sentinel-1 (ESA) via Google Earth Engine. Services and footpaths: OpenStreetMap contributors (ODbL). "
          "Timetables: Prasarana GTFS via data.gov.my (CC BY 4.0). Population: Kontur 2023-11-01 (CC BY 4.0). "
          "Basemap: \u00a9 CARTO, \u00a9 OpenStreetMap contributors.")      # <-
LINKS = [("population.html", "See where people live (Stage 1)"),             # <- links at the bottom of the panel
         ("index.html", "See the 15-minute walking map (Stage 2)"),
         ("buses.html", "See a day of buses (Stage 3)"),
         ("floods.html", "See the 2021 floods (Stage 4)"),
         ("case-study.html", "Read the case study")]
LINK_TO_THIS_PAGE = "Read the story: flooded in 2021, served today? (Stage 5)"   # <- link added to the other pages
NEXT_LABEL, BACK_LABEL = "Next \u2192", "\u2190 Back"                      # <- buttons

# ---- Cameras ----
VALLEY_VIEW = dict(latitude=3.0, longitude=101.6, zoom=9.8, pitch=40, bearing=-15)   # <- whole valley
FLY_MS = 2500               # <- camera flight time between steps, milliseconds

# ---- The story: one entry per step ----
# mode: "population" (all hexagons by people), "flood" (flood tiers), "served" (flood hexagons: poorly served or not),
#       "score" (all hexagons by walking score), "explore" (free map with buttons)
# place: a story place name from 14combine (draws its circle), or None
# view: camera; None = whole valley
STEPS = [                                                                    # <-
     dict(title="The Klang Valley", mode="population", place=None, view=dict(latitude=3.0, longitude=101.8, zoom=9.6, pitch=40, bearing=-15),
         text="About <b>{valley_people}</b> people live in the 4,003 hexagons of the Klang Valley, each about 0.74 km\u00b2. "
              "Taller and darker means more people.<br><br>This story asks one question: "
              "<b>how well served are the areas that flooded in December 2021, today?</b>"),
    dict(title="December 2021", mode="flood", place=None, view=dict(latitude=2.94, longitude=101.6, zoom=10.6, pitch=40, bearing=-15),
         text="Days of heavy rain from 16 December 2021 flooded large parts of Selangor and Kuala Lumpur. "
              "Radar images from 20 and 22 December show a flood signal in <b>{flooded_hex} hexagons</b> where "
              "<b>{flooded_people}</b> people live: {strong_hex} with a strong signal (both days) and "
              "{possible_hex} with a possible signal (one day).<br><br>This is a minimum: the images came 2 to 4 days "
              "after the peak, and radar misses water between tall buildings."),
    dict(title="Taman Sri Muda, Shah Alam", mode="served", place="Taman Sri Muda",
         view=dict(latitude=3.028, longitude=101.537, zoom=12.3, pitch=45, bearing=-20),
         text="The whole township flooded from 19 to 21 December 2021, and boats were the only way in. It has the "
              "strongest radar signal on the map ({tsm_strong} strong hexagons).<br><br>Today, <b>{tsm_poor}</b> of the "
              "{tsm_people} people in its flagged hexagons are poorly served: their average walking score is "
              "<b>{tsm_score} of 5</b>, with about {tsm_buses} Rapid KL buses an hour nearby at the morning peak."),
    dict(title="Meru and Bukit Raja, Klang", mode="served", place="Meru and Bukit Raja, Klang",
         view=dict(latitude=3.128, longitude=101.435, zoom=12.1, pitch=45, bearing=-10),
         text="On 18 December the Meru to Bukit Raja road was under 2 to 3 m of water.<br><br>"
              "Today, <b>{meru_poor}</b> of the {meru_people} people in its flagged hexagons are poorly served: "
              "an average walking score of <b>{meru_score} of 5</b> and about {meru_buses} Rapid KL buses an hour "
              "at the morning peak. Smart Selangor buses also run in Klang but have no open timetable, so the real "
              "bus service is better than shown."),
    dict(title="Dengkil and Salak Tinggi, Sepang", mode="served", place="Dengkil and Salak Tinggi",
         view=dict(latitude=2.840, longitude=101.685, zoom=11.2, pitch=40, bearing=-15),
         text="Dengkil was one of the hardest-hit places: over 3,500 homes were reported submerged.<br><br>"
              "Fewer people live here ({dengkil_people} in the flagged hexagons). Their walking scores are higher "
              "(average {dengkil_score} of 5), so only <b>{dengkil_poor}</b> count as poorly served, but no Rapid KL "
              "bus stops near them at the morning peak."),
    dict(title="Central Kuala Lumpur: the contrast", mode="score", place="Central Kuala Lumpur",
         view=dict(latitude=3.150, longitude=101.695, zoom=12.4, pitch=45, bearing=-20),
         text="Dataran Merdeka and Jalan Masjid India were reported flooded around 18 December, yet radar flags almost "
              "nothing here (at most {klcentre_max} of a hexagon): it cannot see water between tall buildings, and much "
              "had drained by 20 December.<br><br>The colours here show the walking score. The {klcentre_circle_people} "
              "people within 2 km have an average of <b>{klcentre_circle_score} of 5</b> and about "
              "<b>{klcentre_circle_buses} buses an hour</b> nearby: some of the best served places in the valley."),
    dict(title="The answer", mode="served", place=None, view=None,
         text="Across all flood-signal hexagons, <b>{flooded_poor}</b> of people are poorly served, close to the "
              "valley-wide {valley_poor}. But where the signal is strong, the places most surely flooded, it is "
              "<b>{strong_poor}</b>.<br><br>By district: <b>{klang_poor}</b> of people in Klang's flood-signal hexagons "
              "are poorly served, against {kl_poor} in Kuala Lumpur's. The places that flooded worst are mostly on the "
              "outer, less served side of the valley."),
    dict(title="Explore the map", mode="explore", place=None, view=None,
         text="Move around and hover over a hexagon (tap on phones) to see its flood signal, walking minutes to each "
              "service and buses an hour. The buttons change what the flood-signal hexagons show."),
]

# ---- Colours ----
STRONG_COLOUR = "#08306b"       # <- strong flood signal (Stage 4)
POSSIBLE_COLOUR = "#4292c6"     # <- possible flood signal (Stage 4)
POOR_COLOUR = "#b2182b"         # <- poorly served
SERVED_COLOUR = "#2166ac"       # <- not poorly served
FAINT_COLOUR = [190, 190, 190, 55]   # <- hexagons outside the focus (RGB + opacity 0 to 255), drawn flat
POP_BREAKS = [500, 1000, 2500, 5000, 10000]                                  # <- Stage 1 classes
POP_COLOURS = ["#FFF0EA", "#FFCBB7", "#E57A7A", "#9B4174", "#532363", "#1D0E2B"]   # <- Stage 1 colours
SCORE_COLOURS = ["#b2182b", "#e0745a", "#f6c2a6", "#c5dbea", "#6aa6cf", "#2166ac"]  # <- Stage 2 colours, score 0 to 5
PLACE_COLOUR = [255, 111, 0, 255]    # <- story place circles (orange, as the Stage 4 markers)
PLACE_LINE_PX = 3                    # <-

# ---- Map (same as Stages 1, 2 and 4) ----
MAP_STYLE = "https://basemaps.cartocdn.com/gl/voyager-gl-style/style.json"   # <- road map
ELEVATION_SCALE = 0.3       # <- column height = people x this
COVERAGE = 0.88             # <-
OPACITY = 0.92              # <-
DISTRICT_LINE = [60, 60, 60, 140]   # <-

# ---- Panel (same as the other pages) ----
PANEL_BG = "rgba(17, 17, 20, 0.85)"   # <-
PANEL_TEXT = "#f2f2f2"                # <-
PANEL_MUTED = "#a8a8b3"               # <-
LINK_COLOUR = "#8ec1ff"               # <-
PANEL_WIDTH = 400                     # <- px
TITLE_SIZE = 20                       # <- px

# ---- Phones and tablets ----
PHONE_MODE = True           # <- flat map, panel at the bottom
PHONE_MAX_WIDTH = 768       # <-
PHONE_ZOOM_DROP = 1.4       # <- phones zoom out this much from each step's camera
PHONE_PANEL_HEIGHT = 0.42   # <- share of the screen height the panel may use
PHONE_NOTE = "On phones the map is flat. Open it on a computer for the 3D view."   # <-

In [114]:
import json, shutil
import numpy as np
import pandas as pd

hexes = pd.read_parquet(PROCESSED / "hex_story.parquet")
summary = json.loads((PROCESSED / "story_summary.json").read_text(encoding="utf-8"))
districts = json.loads((PROCESSED / "districts.geojson").read_text(encoding="utf-8"))
print(f"{len(hexes):,} hexagons | {len(summary['places'])} story places | {len(districts['features'])} districts")

4,003 hexagons | 4 story places | 7 districts


## Step 1: the numbers for the story texts

In [115]:
def about(x):
    if x >= 1_000_000:
        return f"{x / 1e6:.1f} million"
    return f"about {int(round(x, -3)):,}" if x >= 1000 else f"{int(x):,}"

def pct(x):
    return "none" if x is None else f"{100 * x:.0f}%"

G = summary["groups"]
N = {
    "valley_people": about(G["valley"]["people"]),
    "flooded_hex": G["flooded"]["hexagons"], "flooded_people": about(G["flooded"]["people"]),
    "strong_hex": G["strong"]["hexagons"], "strong_people": about(G["strong"]["people"]),
    "possible_hex": G["possible"]["hexagons"], "possible_people": about(G["possible"]["people"]),
    "strong_poor": pct(G["strong"]["poor_share"]), "possible_poor": pct(G["possible"]["poor_share"]),
    "flooded_poor": pct(G["flooded"]["poor_share"]), "valley_poor": pct(G["valley"]["poor_share"]),
}
for key, district in [("klang", "Klang"), ("kl", "W.P. Kuala Lumpur")]:
    s = summary["by_district"][district]
    N[f"{key}_poor"], N[f"{key}_people"] = pct(s["poor_share"]), about(s["people"])

PLACE_KEYS = {"Taman Sri Muda": "tsm", "Meru and Bukit Raja, Klang": "meru",
              "Dengkil and Salak Tinggi": "dengkil", "Central Kuala Lumpur": "klcentre"}   # name in 14combine -> short key
for p in summary["places"]:
    k = PLACE_KEYS.get(p["name"])
    if not k:
        continue
    N[f"{k}_people"] = about(p.get("people", 0))
    N[f"{k}_poor"] = pct(p.get("poor_share"))
    N[f"{k}_score"] = "no" if p.get("avg_score") is None else f"{p['avg_score']:.1f}"
    N[f"{k}_buses"] = "no" if p.get("avg_peak_buses") is None else f"{p['avg_peak_buses']:.1f}"
    N[f"{k}_strong"] = p.get("strong_hexagons", 0)
    N[f"{k}_max"] = pct(p.get("max_flagged_share"))
    N[f"{k}_circle_people"] = about(p["circle"]["people"])
    N[f"{k}_circle_score"] = f"{p['circle']['avg_score']:.1f}"
    N[f"{k}_circle_buses"] = round(p["circle"]["avg_peak_buses"])
for k, v in N.items():
    print(f"{{{k}}}".ljust(26), v)

{valley_people}            8.3 million
{flooded_hex}              213
{flooded_people}           about 988,000
{strong_hex}               18
{strong_people}            about 47,000
{possible_hex}             195
{possible_people}          about 941,000
{strong_poor}              69%
{possible_poor}            40%
{flooded_poor}             42%
{valley_poor}              39%
{klang_poor}               81%
{klang_people}             about 242,000
{kl_poor}                  6%
{kl_people}                about 241,000
{tsm_people}               about 39,000
{tsm_poor}                 58%
{tsm_score}                1.5
{tsm_buses}                2.4
{tsm_strong}               6
{tsm_max}                  26%
{tsm_circle_people}        about 58,000
{tsm_circle_score}         1.5
{tsm_circle_buses}         2
{meru_people}              about 36,000
{meru_poor}                92%
{meru_score}               1.0
{meru_buses}               0.2
{meru_strong}              6
{meru_max}               

## Step 2: fill in the texts and pack the data

In [116]:
names = {p["name"]: p for p in summary["places"]}
steps = []
for i, s in enumerate(STEPS):
    try:
        text = s["text"].format(**N)
    except KeyError as e:
        raise KeyError(f"Step {i + 1} ({s['title']}) uses {e}, which is not in the list printed above")
    if s["place"] and s["place"] not in names:
        raise KeyError(f"Step {i + 1}: place '{s['place']}' is not in STORY_PLACES of 14combine")
    steps.append({"title": s["title"], "text": text, "mode": s["mode"], "place": s["place"],
                  "view": s["view"] or VALLEY_VIEW})
places = [{"name": p["name"], "lat": p["lat"], "lon": p["lon"], "r": p["radius_km"] * 1000} for p in summary["places"]]

TIER = {"none": 0, "possible": 1, "strong": 2}
def m(v):
    return -1 if pd.isna(v) else round(float(v), 1)
rows = [[r.h3, int(r.population), TIER[r.tier], round(100 * r.F_a, 1), round(100 * r.F_b, 1), int(r.score),
         m(r.healthcare_min), m(r.school_min), m(r.park_min), m(r.grocery_min), m(r.rail_min),
         round(float(r.weekday_peak), 1), int(bool(r.rapid_gap)), int(bool(r.poor)), r.district.replace("W.P. ", "")]
        for r in hexes.itertuples()]
legend_counts = {
    "strong": f"{G['strong']['hexagons']} hexagons, {about(G['strong']['people'])} people",
    "possible": f"{G['possible']['hexagons']} hexagons, {about(G['possible']['people'])} people",
    "poor": f"{about(G['flooded']['poor_people'])} people ({pct(G['flooded']['poor_share'])})",
    "served": f"{about(G['flooded']['people'] - G['flooded']['poor_people'])} people",
}
print(len(rows), "hexagon rows;", len(steps), "steps")

4003 hexagon rows; 8 steps


## Step 3: build the page

The page itself (HTML and JavaScript) is in the next cell as a template. Settings from Section 0 are passed in
as one block called `CFG`; there is normally no need to edit the template.

In [117]:
TEMPLATE = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__PAGE_TITLE__</title>
<script src="https://unpkg.com/h3-js@4.4.0/dist/h3-js.umd.js"></script>
<script src="https://unpkg.com/deck.gl@9.3.11/dist.min.js"></script>
<script src="https://unpkg.com/maplibre-gl@5.14.0/dist/maplibre-gl.js"></script>
<link href="https://unpkg.com/maplibre-gl@5.14.0/dist/maplibre-gl.css" rel="stylesheet">
<style>
  html, body { margin:0; padding:0; height:100%; overflow:hidden; font-family:"Segoe UI", Arial, sans-serif; }
  #map { position:absolute; inset:0; }
  #panel { position:absolute; top:12px; left:12px; z-index:10; width:min(var(--panel-w), calc(100vw - 48px));
    max-height:calc(100vh - 48px); overflow-y:auto; box-sizing:border-box;
    background:var(--panel-bg); color:var(--panel-text); padding:14px 16px; border-radius:8px; font-size:13px; line-height:1.5; }
  .top { display:flex; justify-content:space-between; align-items:baseline; gap:8px; }
  #panel h1 { font-size:var(--title-size); margin:0 0 6px; font-weight:600; }
  #count { color:var(--panel-muted); font-size:12px; white-space:nowrap; }
  #panel h2 { font-size:17px; margin:8px 0 4px; font-weight:600; color:#fff; }
  #stepText { font-size:14px; }
  #stepText b { color:#fff; }
  .row { display:flex; align-items:center; gap:8px; margin:3px 0; }
  .sw { width:14px; height:14px; border-radius:3px; flex:none; }
  .sw.ring { border-radius:50%; background:transparent !important; box-sizing:border-box; }
  .lab { width:150px; flex:none; }
  .pct { color:var(--panel-muted); }
  #legend { margin-top:10px; padding-top:8px; border-top:1px solid rgba(255,255,255,0.15); }
  .small { color:var(--panel-muted); font-size:11px; margin-top:6px; }
  .nav { display:flex; align-items:center; gap:8px; margin:12px 0 4px; }
  .nav button { background:rgba(255,255,255,0.15); color:var(--panel-text); border:0; border-radius:6px; padding:7px 14px;
    font:inherit; cursor:pointer; }
  .nav button.primary { background:var(--link); color:#0b1a2c; font-weight:600; }
  .nav button:disabled { opacity:0.35; cursor:default; }
  .nav button:focus-visible, .seg button:focus-visible, #dots button:focus-visible { outline:2px solid #fff; outline-offset:1px; }
  #dots { display:flex; gap:5px; flex:1; justify-content:center; }
  #dots button { width:9px; height:9px; padding:0; border-radius:50%; border:0; background:rgba(255,255,255,0.3); cursor:pointer; }
  #dots button.on { background:var(--link); }
  #skip { font-size:12px; }
  .seg { display:flex; margin:8px 0 2px; border-radius:6px; overflow:hidden; border:1px solid rgba(255,255,255,0.25); }
  .seg button { flex:1; background:transparent; color:var(--panel-text); border:0; padding:6px 6px; font:inherit; font-size:12px; cursor:pointer; }
  .seg button.on { background:rgba(255,255,255,0.18); font-weight:600; }
  .lnk { font-size:12px; margin-top:6px; }
  #panel a { color:var(--link); }
  #more summary { cursor:pointer; color:var(--link); }
  .phone-only { display:none; }
  /* Phones: panel at the bottom, flat map */
  html.phone #panel { top:auto; bottom:0; left:0; width:100vw; max-height:var(--phone-h); border-radius:12px 12px 0 0;
    padding:10px 14px 12px; font-size:12px; }
  html.phone #panel h1 { font-size:15px; margin:0; }
  html.phone #panel h2 { font-size:15px; margin:4px 0 2px; }
  html.phone #stepText { font-size:13px; }
  html.phone .lab { width:120px; }
  html.phone .phone-only { display:block; }
  html.phone .nav { position:sticky; bottom:-12px; margin:8px -14px -12px; padding:8px 14px 10px;
    background:#1b1b20; border-top:1px solid rgba(255,255,255,0.12); z-index:2; }
  html.phone .desk-only { display:none !important; }
  #err { position:absolute; left:50%; top:50%; transform:translate(-50%,-50%); z-index:20; background:var(--panel-bg);
    color:var(--panel-text); padding:10px 16px; border-radius:8px; font-size:14px; display:none; }
</style>
</head>
<body>
<div id="map"></div>
<div id="err"></div>
<div id="panel">
  <div class="top"><h1 id="title"></h1><span id="count"></span></div>
  <h2 id="stepTitle"></h2>
  <div id="stepText"></div>
  <div id="exploreBox" style="display:none">
    <div class="small" style="margin-top:10px">Colour the flood-signal hexagons by:</div>
    <div class="seg" id="exploreSeg" role="group" aria-label="Colour the flood-signal hexagons by">
      <button data-e="served">Poorly served</button><button data-e="flood">Flood signal</button><button data-e="score">Walking score</button>
    </div>
  </div>
  <div id="legend"></div>
  <div class="nav">
    <button id="back"></button>
    <div id="dots" role="group" aria-label="Story steps"></div>
    <button id="next" class="primary"></button>
  </div>
  <div class="lnk"><a href="#" id="skip">Skip to the map</a></div>
  <div id="caveats"></div>
  <div class="small phone-only" id="phoneNote"></div>
  <details class="small" id="more"><summary id="moreTitle"></summary><div id="moreNotes"></div><div class="small" id="credit"></div></details>
  <div id="links"></div>
</div>

<script>
const CFG = __CONFIG__;
const DATA = __DATA__;
const root = document.documentElement.style;
root.setProperty("--panel-bg", CFG.panel.bg); root.setProperty("--panel-text", CFG.panel.text);
root.setProperty("--panel-muted", CFG.panel.muted); root.setProperty("--link", CFG.panel.link);
root.setProperty("--panel-w", CFG.panel.width + "px"); root.setProperty("--title-size", CFG.panel.titleSize + "px");
root.setProperty("--phone-h", Math.round(CFG.phone.panelHeight * 100) + "vh");

const isPhone = CFG.phone.enabled && (matchMedia(`(max-width: ${CFG.phone.maxWidth}px)`).matches ||
                                      matchMedia("(pointer: coarse)").matches);
if (isPhone) document.documentElement.classList.add("phone");
const $ = id => document.getElementById(id);

// ---------- colours ----------
const hexRGB = h => [parseInt(h.slice(1, 3), 16), parseInt(h.slice(3, 5), 16), parseInt(h.slice(5, 7), 16)];
const C = CFG.colours, A = Math.round(CFG.map.opacity * 255);
const RGB = { strong: hexRGB(C.strong), possible: hexRGB(C.possible), poor: hexRGB(C.poor), served: hexRGB(C.served) };
const POP = C.pop.map(hexRGB), SCORE = C.score.map(hexRGB);
function popClass(v) { let k = 0; for (let i = 0; i < C.popBreaks.length; i++) if (v >= C.popBreaks[i]) k = i + 1; return k; }
// row: 0 h3, 1 people, 2 tier (0 none, 1 possible, 2 strong), 3 % flagged 20 Dec, 4 % flagged 22 Dec, 5 walking score,
//      6 clinic, 7 school, 8 park, 9 grocery, 10 rail (minutes, -1 = over 75), 11 buses an hour (weekday peak),
//      12 Rapid KL gap, 13 poorly served, 14 district

// ---------- state ----------
const steps = CFG.steps;
const state = { step: 0, explore: "served" };
const fromHash = parseInt((location.hash.match(/step=(\d+)/) || [])[1], 10);
if (fromHash >= 1 && fromHash <= steps.length) state.step = fromHash - 1;
const mode = () => steps[state.step].mode;
const colourBy = () => mode() === "explore" ? state.explore : mode();

function inFocus(r) { const m = colourBy(); return m === "population" || m === "score" || r[2] > 0; }
function fill(r) {
  if (!inFocus(r)) return C.faint;
  switch (colourBy()) {
    case "population": return [...POP[popClass(r[1])], A];
    case "score": return [...SCORE[r[5]], A];
    case "flood": return [...(r[2] === 2 ? RGB.strong : RGB.possible), A];
    default: return [...(r[13] ? RGB.poor : RGB.served), A];
  }
}

// ---------- camera ----------
function viewFor(v) {
  if (!isPhone) {
    // The panel covers the left of the screen: move the camera west so the place shows in the visible part.
    const shiftPx = Math.min(CFG.panel.width + 24, window.innerWidth * 0.4) / 2;
    return { ...v, longitude: v.longitude - shiftPx * 360 / (512 * Math.pow(2, v.zoom)) };
  }
  const z = v.zoom - CFG.phone.zoomDrop;
  // The panel covers the bottom of the screen: move the camera south so the place shows in the visible part.
  const shiftPx = CFG.phone.panelHeight * window.innerHeight / 2;
  return { latitude: v.latitude - shiftPx * 360 / (512 * Math.pow(2, z)), longitude: v.longitude, zoom: z, pitch: 0, bearing: 0 };
}

// ---------- tooltip ----------
const mins = v => v < 0 ? "over 75" : String(Math.round(v));
const TIER_NAME = ["No radar signal", "Possible flood signal", "Strong flood signal"];
function tooltip({ object, layer }) {
  if (!object || !layer || layer.id !== "hex") return null;
  const r = object, per = v => v >= 1 ? `${v.toFixed(v < 10 ? 1 : 0)} an hour` : (v > 0 ? "less than 1 an hour" : "none");
  return { html: `<b>${r[14]}</b> &middot; ${r[1].toLocaleString()} people<br/>` +
    `<b>${TIER_NAME[r[2]]}</b> (${r[3].toFixed(1)}% of the hexagon on 20 Dec, ${r[4].toFixed(1)}% on 22 Dec)<br/>` +
    `<b>Walking score ${r[5]} of 5</b>: clinic ${mins(r[6])}, school ${mins(r[7])}, park ${mins(r[8])}, ` +
    `grocery ${mins(r[9])}, rail ${mins(r[10])} min<br/>` +
    `Rapid KL and MRT feeder buses within 400 m: ${per(r[11])} (weekday 7 to 9 am)` +
    (r[13] ? `<br/><b>Poorly served</b>` : (r[12] ? `<br/>Gap in the Rapid KL network` : "")),
    style: CFG.tooltipStyle };
}

// ---------- map ----------
const deckgl = new deck.DeckGL({
  container: "map",
  mapStyle: CFG.map.style,
  initialViewState: viewFor(steps[state.step].view),
  controller: true,
  useDevicePixels: !isPhone,
  getTooltip: tooltip,
  onError: e => console.error(e),
});

function layers() {
  const s = steps[state.step], m = mode();
  const showPlaces = s.place ? CFG.places.filter(p => p.name === s.place)
                             : (m === "flood" || m === "served" || m === "explore" ? CFG.places : []);
  return [
    new deck.GeoJsonLayer({ id: "districts", data: DATA.districts, stroked: true, filled: false,
      getLineColor: CFG.map.districtLine, lineWidthMinPixels: 1 }),
    new deck.H3HexagonLayer({ id: "hex", data: DATA.rows, getHexagon: r => r[0],
      getFillColor: fill, extruded: !isPhone, getElevation: r => inFocus(r) ? r[1] : 0,
      elevationScale: CFG.map.elevationScale, coverage: CFG.map.coverage, pickable: true, autoHighlight: !isPhone,
      transitions: { getFillColor: 600, getElevation: 600 },
      updateTriggers: { getFillColor: [m, state.explore], getElevation: [m, state.explore] } }),
    new deck.ScatterplotLayer({ id: "places", data: showPlaces, getPosition: p => [p.lon, p.lat], getRadius: p => p.r,
      radiusUnits: "meters", stroked: true, filled: false, getLineColor: C.place, lineWidthUnits: "pixels",
      getLineWidth: C.placeLine, parameters: { depthCompare: "always" } }),
    new deck.TextLayer({ id: "placeNames", data: s.place ? [] : showPlaces, getText: p => p.name,
      getPosition: p => [p.lon, p.lat + p.r / 111000], getSize: isPhone ? 11 : 13, getColor: [30, 30, 30, 255],
      getTextAnchor: "middle", getAlignmentBaseline: "bottom", background: true, getBackgroundColor: [255, 255, 255, 220],
      backgroundPadding: [4, 2], fontFamily: "Segoe UI, Arial, sans-serif", parameters: { depthCompare: "always" } }),
  ];
}

// ---------- panel ----------
const fmtN = n => n.toLocaleString();
function legendHTML() {
  const L = CFG.legendCounts, m = colourBy();
  const row = (colour, lab, txt, ring) => `<div class="row"><span class="sw${ring ? " ring" : ""}" style="${ring ?
    `border:3px solid ${colour}` : `background:${colour}`}"></span><span class="lab">${lab}</span><span class="pct">${txt || ""}</span></div>`;
  const faint = `rgba(${C.faint.slice(0, 3).join(",")},0.6)`;
  const ringRow = steps[state.step].place || ["flood", "served", "explore"].includes(mode())
    ? row(`rgb(${C.place.slice(0, 3).join(",")})`, "Story place", "", true) : "";
  if (m === "population") {
    const b = C.popBreaks, labs = [`under ${fmtN(b[0])}`, ...b.slice(0, -1).map((x, i) => `${fmtN(x)} to ${fmtN(b[i + 1])}`), `${fmtN(b[b.length - 1])} or more`];
    return `<div class="small" style="margin:0 0 4px">People per hexagon <span class="desk-only">(height too)</span></div>` +
      labs.map((l, i) => [l, C.pop[i]]).reverse().map(([l, c]) => row(c, l)).join("");
  }
  if (m === "score")
    return `<div class="small" style="margin:0 0 4px">Everyday services within 15 minutes' walk <span class="desk-only">(height: people)</span></div>` +
      [5, 4, 3, 2, 1, 0].map(k => row(C.score[k], `${k} of 5`)).join("") + ringRow;
  if (m === "flood")
    return row(C.strong, "Strong signal (both days)", L.strong) + row(C.possible, "Possible signal (one day)", L.possible) +
      row(faint, "No radar signal", "does not mean it stayed dry") + ringRow;
  return `<div class="small" style="margin:0 0 4px">Flood-signal hexagons today <span class="desk-only">(height: people)</span></div>` +
    row(C.poor, "Poorly served", L.poor) + row(C.served, "Not poorly served", L.served) +
    row(faint, "No flood signal", "") + ringRow + `<div class="small">${CFG.poorNote}</div>`;
}
function exploreLegend() {
  if (state.explore === "score") return `<div class="small" style="margin:0 0 4px">Flood-signal hexagons: services within 15 minutes' walk</div>` +
    [5, 4, 3, 2, 1, 0].map(k => `<div class="row"><span class="sw" style="background:${C.score[k]}"></span><span class="lab">${k} of 5</span></div>`).join("");
  return legendHTML();
}

$("title").textContent = CFG.title;
$("back").textContent = CFG.back; $("next").textContent = CFG.next;
$("back").setAttribute("aria-label", "Previous step"); $("next").setAttribute("aria-label", "Next step");
$("caveats").innerHTML = CFG.caveats.map(t => `<div class="small">${t}</div>`).join("");
$("phoneNote").textContent = CFG.phone.note;
$("moreTitle").textContent = CFG.moreTitle;
$("moreNotes").innerHTML = CFG.moreNotes.map(t => `<div class="small">${t}</div>`).join("");
$("credit").textContent = CFG.credit;
$("links").innerHTML = CFG.links.map(([h, t]) => `<div class="lnk"><a href="${h}">${t}</a></div>`).join("");
$("dots").innerHTML = steps.map((s, i) => `<button data-i="${i}" aria-label="Step ${i + 1}: ${s.title}" title="${s.title}"></button>`).join("");

function show(i, fly = true) {
  state.step = Math.max(0, Math.min(steps.length - 1, i));
  const s = steps[state.step];
  $("count").textContent = `${state.step + 1} of ${steps.length}`;
  $("stepTitle").textContent = s.title;
  $("stepText").innerHTML = s.text;
  $("exploreBox").style.display = s.mode === "explore" ? "block" : "none";
  document.querySelectorAll("#exploreSeg button").forEach(b => b.classList.toggle("on", b.dataset.e === state.explore));
  $("legend").innerHTML = s.mode === "explore" ? exploreLegend() : legendHTML();
  $("back").disabled = state.step === 0;
  $("next").disabled = state.step === steps.length - 1;
  $("skip").style.display = state.step === steps.length - 1 ? "none" : "inline";
  document.querySelectorAll("#dots button").forEach((b, k) => b.classList.toggle("on", k === state.step));
  try { history.replaceState(null, "", `#step=${state.step + 1}`); } catch (e) {}
  const props = { layers: layers() };
  if (fly) props.initialViewState = { ...viewFor(s.view), transitionDuration: CFG.flyMs,
                                      transitionInterpolator: new deck.FlyToInterpolator() };
  deckgl.setProps(props);
  $("panel").scrollTop = 0;
}
$("back").onclick = () => show(state.step - 1);
$("next").onclick = () => show(state.step + 1);
$("skip").onclick = e => { e.preventDefault(); show(steps.length - 1); };
document.querySelectorAll("#dots button").forEach(b => b.onclick = () => show(+b.dataset.i));
document.querySelectorAll("#exploreSeg button").forEach(b => b.onclick = () => { state.explore = b.dataset.e; show(state.step, false); });
document.addEventListener("keydown", e => {
  if (e.target.tagName === "INPUT" || e.target.tagName === "SELECT") return;
  if (e.key === "ArrowRight" || e.key === "PageDown") { e.preventDefault(); show(state.step + 1); }
  if (e.key === "ArrowLeft" || e.key === "PageUp") { e.preventDefault(); show(state.step - 1); }
});
window.addEventListener("hashchange", () => {
  const k = parseInt((location.hash.match(/step=(\d+)/) || [])[1], 10);
  if (k >= 1 && k <= steps.length && k - 1 !== state.step) show(k - 1);
});
try { show(state.step, false); }
catch (err) { $("err").style.display = "block"; $("err").textContent = "Could not draw the map: " + err.message; }
</script>
</body>
</html>
'''

In [118]:
config = {
    "title": TITLE, "steps": steps, "places": places, "poorNote": POOR_NOTE, "caveats": CAVEATS,
    "moreTitle": MORE_NOTES_TITLE, "moreNotes": MORE_NOTES, "credit": CREDIT, "links": LINKS,
    "next": NEXT_LABEL, "back": BACK_LABEL, "flyMs": FLY_MS, "legendCounts": legend_counts,
    "colours": {"strong": STRONG_COLOUR, "possible": POSSIBLE_COLOUR, "poor": POOR_COLOUR, "served": SERVED_COLOUR,
                "faint": FAINT_COLOUR, "popBreaks": POP_BREAKS, "pop": POP_COLOURS, "score": SCORE_COLOURS,
                "place": PLACE_COLOUR, "placeLine": PLACE_LINE_PX},
    "map": {"style": MAP_STYLE, "elevationScale": ELEVATION_SCALE, "coverage": COVERAGE, "opacity": OPACITY,
            "districtLine": DISTRICT_LINE},
    "panel": {"bg": PANEL_BG, "text": PANEL_TEXT, "muted": PANEL_MUTED, "link": LINK_COLOUR,
              "width": PANEL_WIDTH, "titleSize": TITLE_SIZE},
    "phone": {"enabled": PHONE_MODE, "maxWidth": PHONE_MAX_WIDTH, "zoomDrop": PHONE_ZOOM_DROP,
              "panelHeight": PHONE_PANEL_HEIGHT, "note": PHONE_NOTE},
    "tooltipStyle": {"backgroundColor": "#1b1b22", "color": "#f2f2f2", "fontSize": "13px",
                     "fontFamily": "Segoe UI, Arial, sans-serif", "padding": "8px 10px", "lineHeight": "1.5"},
}
assert len(POP_COLOURS) == len(POP_BREAKS) + 1, "POP_COLOURS needs one more colour than POP_BREAKS"
assert len(SCORE_COLOURS) == 6, "SCORE_COLOURS needs 6 colours (score 0 to 5)"
data = {"rows": rows, "districts": districts}
html = (TEMPLATE.replace("__PAGE_TITLE__", PAGE_TITLE)
                .replace("__CONFIG__", json.dumps(config, ensure_ascii=False))
                .replace("__DATA__", json.dumps(data, separators=(",", ":"), ensure_ascii=False)))
DOCS.mkdir(exist_ok=True)
OUT_HTML.write_text(html, encoding="utf-8")
print("Saved", OUT_HTML.name, "|", round(OUT_HTML.stat().st_size / 1024), "KB")

Saved story.html | 344 KB


## Step 4: link to this page from the other four pages

Adds one link to `docs/population.html`, `docs/index.html` and `docs/floods.html` (after their last link) and to the
links list inside `docs/buses.html` (only once). Run this step again after re-running 06buildaccess, 09buildbuses or
13buildfloods, because they rewrite those pages.

In [119]:
link = (f'<div class="lnk" style="font-size:12px; margin-top:8px;">'
        f'<a href="story.html" style="color:{LINK_COLOUR}">{LINK_TO_THIS_PAGE}</a></div>')
for name in ["population.html", "index.html", "floods.html"]:
    f = DOCS / name
    if not f.exists():
        print(name, "not found"); continue
    s = f.read_text(encoding="utf-8")
    if 'href="story.html"' in s:
        print(name, "already links to story.html"); continue
    i = s.rfind('<div class="lnk"')
    if i < 0:
        print(name, "no link block found, not changed"); continue
    j = s.find("</div>", i) + len("</div>")
    f.write_text(s[:j] + "\n  " + link + s[j:], encoding="utf-8")
    print(name, "link added")

f = DOCS / "buses.html"
if f.exists():
    s = f.read_text(encoding="utf-8")
    old = '["floods.html", "See the 2021 floods (Stage 4)"]'
    if '"story.html"' in s:
        print("buses.html already links to story.html")
    elif old in s:
        f.write_text(s.replace(old, old + ", " + json.dumps(["story.html", LINK_TO_THIS_PAGE], ensure_ascii=False), 1),
                     encoding="utf-8")
        print("buses.html link added")
    else:
        print("buses.html: Stage 4 link entry not found, not changed (run Step 4 of 13buildfloods first)")

population.html already links to story.html
index.html already links to story.html
floods.html already links to story.html
buses.html already links to story.html


## Step 5: open the page

The data is inside the page, so it opens straight from the folder (no web server needed).

In [120]:
import webbrowser
webbrowser.open(OUT_HTML.resolve().as_uri())

True

___
**END**